In [6]:
import numpy as np
import h5py
import pandas as pd
import ase
import uuid
import datetime
from hdf_tools import *

In [ ]:
def write_hdf(method, code, cell:ase.Atoms, data, authors, path):
    # id = uuid.uuid4()
    id = 'test'
    print(id)

    with h5py.File(f'{path}/{id}.hdf5','w') as f:
        f.attrs.create('system', f'{data['stack']}_' + f'd{data['d']:.1f}_' + f'{data['ntiling']:1}x{data['ntiling']:1}')
        f.attrs.create('formula', cell.get_chemical_formula())
        f.attrs.create('natoms', cell.get_number_of_atoms())
        f.attrs.create('method', method)
        species = list(set(cell.get_chemical_symbols()))
        f.attrs.create('species', species)
        f.attrs.create('creation_date', datetime.datetime.now().isoformat())
        f.attrs.create('uuid', str(id))
        f.attrs.create('author', authors)

        f.create_group('code')
        f['code'].attrs.create('name',code)
        f['code/system'] = ["Illinois Campus Cluster Program", "Oak Ridge Leadership Computing Facility"]

        f['parameters/time_step'] = data['timestep']
        f['parameters/kpoint_grid'] = [4,4,1] # from Krongchon PHYSICAL REVIEW B 108, 235403 (2023)
        # f['parameters/supercell_size'] = ntiling
        # f['parameters/nblocks'] = data['nblocks']

        f['structure/lattice_vectors'] = np.array(cell.get_cell())
        f['structure/positions'] = cell.get_positions()
        f['structure/fractional_positions'] = cell.get_scaled_positions()
        f['structure/pbc'] = cell.get_pbc()

        f['observables/total_energy'] = data['total_energy']
        f['observables/total_energy_error'] = data['total_energy_err']


In [ ]:
fname = './data_energy.csv'
authors = ['Kittithat Krongchon', 'Tawfiqur Rakib', 'Shivesh Pathak', 'Elif Ertekin', 'Harley T. Johnson' ,'Lucas K. Wagner']
raw = pd.read_csv(fname)

# for i in range(raw.shape[0]):
for i in range(1):
    data = raw.iloc[i]
    print(data)

    atoms = get_basis(d=data['d'],disregistry=data['registry'])
    lattice = get_lattice_vectors()
    ase_atoms = ase.Atoms('CCCC', positions=atoms, cell = lattice, pbc=[1,1,0])

    P=np.diag([data['ntiling'],data['ntiling'],1])
    super = ase.build.make_supercell(ase_atoms, P=P)
    print(super.get_number_of_atoms)

    write_hdf('DMC', 'QMCPACK', super, data, authors, path='./')

registry                0.66667
ntiling                       3
d                           3.0
total_energy       -5936.275164
total_energy_err       0.020962
autocorrelation             1.0
acc                    0.990621
walltime            6004.755832
timestep                   0.02
nblocks                    16.0
energy              -659.586129
energy_err             0.002329
stack                        AA
Name: 0, dtype: object
<bound method Atoms.get_number_of_atoms of Atoms(symbols='C36', pbc=[True, True, False], cell=[[7.38, 0.0, 0.0], [3.69, 6.391267479929157, 0.0], [0.0, 0.0, 0.0]])>
f51959ed-19a8-4b75-9757-f28be29f03ed


/tmp/ipykernel_462821/3479789273.py:8: FutureWarning: Please use len(self) or, if your atoms are distributed, self.get_global_number_of_atoms.
  f.attrs.create('natoms', cell.get_number_of_atoms())
